In [5]:
from themoviedb import TMDb

tmdb = TMDb(key="14bb79793229b223948284ea27cf1f39")
# or: tmdb = aioTMDb(key="YOUR_API_KEY", language="pt-BR", region="BR")

In [ ]:
import requests

TMDB_API_KEY = '14bb79793229b223948284ea27cf1f39'
TMDB_BASE_URL = "https://api.themoviedb.org/3"


def fetch_movie_by_tmdb_id(tmdb_id: int, language="en-US"):
    url = f"{TMDB_BASE_URL}/movie/{tmdb_id}"
    params = {
        "api_key": TMDB_API_KEY,
        "language": language,
        "append_to_response": "credits"
    }

    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    data = r.json()

    # --- Parse core fields ---
    overview = data.get("overview")
    release_date = data.get("release_date")
    genres = [g["name"] for g in data.get("genres", [])]

    # --- Parse credits ---
    cast = data.get("credits", {}).get("cast", [])
    crew = data.get("credits", {}).get("crew", [])

    actors = [c["name"] for c in cast]  # ordered by billing
    directors = [c["name"] for c in crew if c["job"] == "Director"]

    return {
        "tmdb_id": tmdb_id,
        "title": data.get("title"),
        "overview": overview,
        "release_date": release_date,
        "genres": genres,
        "actors": actors,
        "directors": directors
    }

In [10]:
movie = fetch_movie_by_tmdb_id(862)  # Toy Story
print(movie["title"])
print(movie["actors"][:5])
print(movie["overview"])

Toy Story
['Tom Hanks', 'Tim Allen', 'Don Rickles', 'Jim Varney', 'Wallace Shawn']
Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.


In [16]:
import os
import csv
import pandas as pd
import time
from tqdm import tqdm


def enrich_movies_with_tmdb(
    movies_path="./data/movies.dat",
    links_path="./data/links.csv",
    output_path="movies_enriched.csv",
    sleep_sec=0.25,
    max_actors=10
):
    # --- Load MovieLens movies ---
    movies = pd.read_csv(
        movies_path,
        sep="::",
        engine="python",
        names=["movieId", "title", "genres"],
        encoding="latin-1"
    )

    # --- Load links ---
    links = pd.read_csv(links_path)

    # --- Join ---
    df = movies.merge(links, on="movieId", how="left")

    # --- Track already processed movieIds ---
    processed_movie_ids = set()
    if os.path.exists(output_path):
        existing = pd.read_csv(output_path)
        processed_movie_ids = set(existing["movieId"].astype(int))

    file_exists = os.path.exists(output_path)

    total = len(df)
    skipped = 0
    processed = 0
    failed = 0

    with open(output_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "movieId",
                "tmdb_id",
                "title",
                "release_date",
                "genres",
                "overview",
                "actors",
                "directors"
            ]
        )

        if not file_exists:
            writer.writeheader()

        for _, row in tqdm(df.iterrows(), total=total, desc="Enriching movies"):
            movie_id = int(row["movieId"])
            tmdb_id = row["tmdbId"]

            if movie_id in processed_movie_ids or pd.isna(tmdb_id):
                skipped += 1
                continue

            try:
                info = fetch_movie_by_tmdb_id(int(tmdb_id))

                writer.writerow({
                    "movieId": movie_id,
                    "tmdb_id": info["tmdb_id"],
                    "title": info["title"],
                    "release_date": info["release_date"],
                    "genres": "|".join(info["genres"]) if info["genres"] else None,
                    "overview": info["overview"],
                    "actors": "|".join(info["actors"][:max_actors]),
                    "directors": "|".join(info["directors"])
                })

                processed += 1
                f.flush()
                time.sleep(sleep_sec)

            except Exception as e:
                failed += 1
                print(f"[WARN] movieId={movie_id}, tmdbId={tmdb_id}: {e}")

    print(
        f"\nDone. processed={processed}, skipped={skipped}, failed={failed}, total={total}"
    )

In [17]:
enrich_movies_with_tmdb(
    movies_path="./data/movies.dat",
    links_path="./data/links.csv",
    output_path="movies_enriched.csv"
)

Enriching movies: 100%|██████████| 3883/3883 [33:01<00:00,  1.96it/s] 


Done. processed=3723, skipped=160, failed=0, total=3883
